# AIC2026 — SigLIP2 gallery from keyframe datasets

Same stills as CLIP (`VIDEO_ID/{map.csv,*.webp}`). **Does not** copy CLIP `embeddings.npy` — new encode.

1. Add secret `AIC2026-PACs-token` (GitHub PAT). Attach **10** keyframe datasets (L21–L30). Edit `KEYFRAME_ROOTS` if Kaggle paths differ.
2. Enable **GPU**. Clone + `pip -r requirements.txt`. `Save Version` → download zips from Output.
3. Writes `features/siglip/Lxx/VIDEO_ID.npy` (row i = map.csv row i, L2-norm) + `model.json`.
4. Merge (same root files as `features/clip/`): `embeddings.npy`, `gallery_map.csv`, `index.faiss`.
5. Zip:
   - **`siglip.zip`** — unpack into `features/siglip/` (per-video + merged gallery)
   - `features-siglip-L21.zip` … `L30` — per-batch (`Lxx/VIDEO_ID.npy`) only

Model: `google/siglip2-so400m-patch16-256` (faster than 384; dim 1152). Resume: skip existing `.npy` with matching row count.


In [ ]:
# GitHub token for private clone
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("AIC2026-PACs-token")


In [ ]:
# Fresh repo checkout in /kaggle/working
!rm -rf /kaggle/working/AIC2026-PACs
!git clone https://{secret_value_0}@github.com/AkiyaNguyen/AIC2026-PACs.git


In [ ]:
# Repo deps, then a recent transformers (SigLIP2).
!pip install -q -r /kaggle/working/AIC2026-PACs/requirements.txt
!pip install -q -U transformers


In [ ]:
# Dataset root = VIDEO_ID/{map.csv, *.webp, ...} (same as uploaded aic2026-keyframes-l**).

KEYFRAME_ROOTS = [
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l21",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l22",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l23",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l24",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l25",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l26",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l27",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l28",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l29",
    "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l30",
    # "/kaggle/input/datasets/akiyanguyen/aic2026-keyframes-l21",
]

In [1]:
import csv
import json
import re
from pathlib import Path

import torch

REPO = Path("/kaggle/working/AIC2026-PACs")
OUT_ROOT = Path("/kaggle/working/features/siglip")
MODEL_ID = "google/siglip2-so400m-patch16-256"
DEVICE = "cuda"  # cuda | cpu
BATCH_SIZE = 16
SKIP_EXISTING = True  # resume if npy rows == map rows
IMAGE_EXTS = {".webp", ".jpg", ".jpeg", ".png"}
BATCH_RE = re.compile(r"^(L\d+)_", re.I)

if not REPO.is_dir():
    raise SystemExit(f"Repo missing (run clone cell): {REPO}")

missing = [d for d in KEYFRAME_ROOTS if not Path(d).is_dir()]
if missing:
    raise SystemExit(f"Not a directory (attach datasets / fix paths): {missing}")

video_dirs: dict[str, Path] = {}
for root in KEYFRAME_ROOTS:
    for map_csv in sorted(Path(root).rglob("map.csv")):
        folder = map_csv.parent
        vid = folder.name
        if vid in video_dirs:
            print(f"warning: duplicate {vid}, keep first {video_dirs[vid]}", flush=True)
            continue
        video_dirs[vid] = folder

if not video_dirs:
    raise SystemExit(f"No VIDEO_ID/map.csv under KEYFRAME_ROOTS={KEYFRAME_ROOTS}")

device = torch.device(
    DEVICE if (DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
)
if DEVICE == "cuda" and device.type == "cpu":
    print("warning: CUDA requested but unavailable; using cpu", flush=True)

print(f"videos={len(video_dirs)}")
print("OUT_ROOT =", OUT_ROOT)
print("MODEL_ID =", MODEL_ID)
print("DEVICE =", device)
print("BATCH_SIZE =", BATCH_SIZE)
if torch.cuda.is_available():
    print("GPU =", torch.cuda.get_device_name(0))


SystemExit: Repo missing (run clone cell): \kaggle\working\AIC2026-PACs

d:\HCMUS_ComputerScience\code\AIC2026\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import numpy as np
from PIL import Image
from transformers import AutoModel, AutoProcessor

OUT_ROOT.mkdir(parents=True, exist_ok=True)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
use_amp = device.type == "cuda"
print(f"model loaded type={type(model).__name__} amp={use_amp}", flush=True)


def sorted_images(folder: Path) -> list[Path]:
    return sorted(
        p
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


def map_nrows(folder: Path) -> int:
    with (folder / "map.csv").open(encoding="utf-8", newline="") as f:
        return sum(1 for _ in csv.DictReader(f))


def batch_id(stem: str) -> str:
    m = BATCH_RE.match(stem)
    return m.group(1) if m else "unknown"


def _as_embed_tensor(out):
    """SigLIP2 AutoModel: tensor, or BaseModelOutputWithPooling."""
    if torch.is_tensor(out):
        return out
    pooled = getattr(out, "pooler_output", None)
    if pooled is not None:
        return pooled
    embeds = getattr(out, "image_embeds", None)
    if embeds is not None:
        return embeds
    hidden = getattr(out, "last_hidden_state", None)
    if hidden is not None:
        return hidden[:, 0]
    raise TypeError(f"Unexpected image features type: {type(out)}")


@torch.inference_mode()
def embed_pils(images: list[Image.Image]) -> np.ndarray:
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if use_amp:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model.get_image_features(**inputs)
    else:
        out = model.get_image_features(**inputs)
    feats = _as_embed_tensor(out).float()
    feats = feats / feats.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return feats.cpu().numpy().astype(np.float32)


def embed_video(folder: Path) -> np.ndarray:
    n_map = map_nrows(folder)
    imgs = sorted_images(folder)
    if n_map == 0 or not imgs:
        raise SystemExit(f"{folder.name}: empty map or no stills")
    if len(imgs) != n_map:
        print(
            f"WARN {folder.name}: images={len(imgs)} map={n_map}; using first {min(len(imgs), n_map)}",
            flush=True,
        )
    n = min(len(imgs), n_map)
    chunks = []
    for start in range(0, n, BATCH_SIZE):
        paths = imgs[start : start + BATCH_SIZE]
        pils = [Image.open(p).convert("RGB") for p in paths]
        try:
            chunks.append(embed_pils(pils))
        finally:
            for im in pils:
                im.close()
    feats = np.concatenate(chunks, axis=0)
    if feats.shape[0] != n:
        raise SystemExit(f"{folder.name}: got {feats.shape[0]} rows, expected {n}")
    return feats


n_ok = 0
n_skip = 0
dim = None
items = sorted(video_dirs.items())
for i, (vid, folder) in enumerate(items, start=1):
    batch = batch_id(vid)
    dest_dir = OUT_ROOT / batch
    dest_dir.mkdir(parents=True, exist_ok=True)
    npy_path = dest_dir / f"{vid}.npy"
    n_map = map_nrows(folder)

    if SKIP_EXISTING and npy_path.is_file():
        existing = np.load(npy_path, mmap_mode="r")
        if existing.ndim == 2 and existing.shape[0] == n_map:
            n_skip += 1
            dim = int(existing.shape[1]) if dim is None else dim
            if i == 1 or i % 50 == 0 or i == len(items):
                print(f"[{i}/{len(items)}] skip {batch}/{vid} rows={n_map}", flush=True)
            continue

    feats = embed_video(folder)
    dim = int(feats.shape[1])
    np.save(npy_path, feats)
    n_ok += 1
    if i == 1 or i % 10 == 0 or i == len(items):
        print(
            f"[{i}/{len(items)}] {batch}/{vid}  rows={feats.shape[0]} dim={feats.shape[1]} → {npy_path}",
            flush=True,
        )

if dim is None:
    raise SystemExit("No embeddings written or skipped")

meta = {
    "model": MODEL_ID,
    "hf_id": MODEL_ID,
    "dim": dim,
    "normalize": True,
    "note": "Rows align with maps/VIDEO_ID.csv and clip/Lxx/VIDEO_ID.npy. Query with the SigLIP2 text tower, not CLIP.",
}
(OUT_ROOT / "model.json").write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
print(f"Done: wrote={n_ok} skipped={n_skip} dim={dim}")
print("model.json =", OUT_ROOT / "model.json")


In [ ]:
# Merge Lxx/*.npy → embeddings.npy + gallery_map.csv + IndexFlatIP (same root as features/clip).
import csv

import faiss
import numpy as np

npy_paths = sorted(p for p in OUT_ROOT.glob("*/*.npy") if p.name != "embeddings.npy")
if not npy_paths:
    npy_paths = sorted(p for p in OUT_ROOT.glob("*.npy") if p.name != "embeddings.npy")
if not npy_paths:
    raise SystemExit(f"No per-video .npy under {OUT_ROOT}")

blocks = []
rows = []
start = 0
dim_m = None
for p in npy_paths:
    block = np.load(p).astype(np.float32, copy=False)
    if block.ndim != 2:
        raise SystemExit(f"Expected 2D {p}, got {block.shape}")
    if dim_m is None:
        dim_m = int(block.shape[1])
    elif int(block.shape[1]) != dim_m:
        raise SystemExit(f"{p}: dim {block.shape[1]} != {dim_m}")
    n = int(block.shape[0])
    blocks.append(block)
    rows.append({"video_id": p.stem, "start_row": start, "n_rows": n})
    start += n
    print(f"  {p.parent.name}/{p.name}  shape={block.shape}", flush=True)

emb = np.concatenate(blocks, axis=0)
out_npy = OUT_ROOT / "embeddings.npy"
out_map = OUT_ROOT / "gallery_map.csv"
np.save(out_npy, emb)
with out_map.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["video_id", "start_row", "n_rows"])
    w.writeheader()
    w.writerows(rows)

index = faiss.IndexFlatIP(int(emb.shape[1]))
index.add(np.ascontiguousarray(emb, dtype=np.float32))
out_idx = OUT_ROOT / "index.faiss"
faiss.write_index(index, str(out_idx))

meta_path = OUT_ROOT / "model.json"
if meta_path.is_file():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    meta["dim"] = int(emb.shape[1])
    meta_path.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

print(f"Wrote {out_npy}  shape={emb.shape}")
print(f"Wrote {out_map}  videos={len(rows)} total_rows={start}")
print(f"Wrote {out_idx}  ntotal={index.ntotal} d={index.d}")
assert start == emb.shape[0]


In [ ]:
# siglip.zip → unpack into features/siglip/
# features-siglip-Lxx.zip → Lxx/VIDEO_ID.npy (+ run-less; model.json only in the full zip)
import zipfile
from IPython.display import FileLink, display

work = Path("/kaggle/working")
root = OUT_ROOT
files = [p for p in root.rglob("*") if p.is_file()]

all_zip = work / "siglip.zip"
all_zip.unlink(missing_ok=True)
with zipfile.ZipFile(all_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in files:
        zf.write(f, arcname=str(f.relative_to(root)))
print(f"Wrote {all_zip.name}: {all_zip.stat().st_size / 1e6:.1f} MB  ({len(files)} files)")
display(FileLink(str(all_zip)))

batches = sorted(p.name for p in root.iterdir() if p.is_dir() and p.name.startswith("L"))
for batch in batches:
    bdir = root / batch
    npys = sorted(bdir.glob("*.npy"))
    if not npys:
        continue
    out = work / f"features-siglip-{batch}.zip"
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in npys:
            zf.write(f, arcname=f"{batch}/{f.name}")
    print(f"Wrote {out.name}: {out.stat().st_size / 1e6:.1f} MB  npy={len(npys)}")
    display(FileLink(str(out)))
